# Limitations And Executive Evidence

This final notebook gives the honest evaluator-facing story: what is real, what is wired, and what remains a limitation.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
Markdown((OUT / 'executive_summary.md').read_text())

# v7 RM/PM Forecast Planning Evidence

## Position

v7 is the WMS planning-grade raw-material / packaging-material forecast layer. It does not claim that v6 FG/bootstrap LightGBM forecasts raw materials.

## Data Truth

- RM/PM demand rows: 10368
- RM/PM demand materials: 288
- Demand window: 2023-02-01 to 2026-01-01
- Existing WMS forecast rows: 6102
- BOM headers/components: 3 headers, 2 component rows
- BOM product-parent coverage: 0.0%

## Model Result

- Selected model: lightgbm_global_rm_pm
- Backtest WAPE: 0.25089219589297024
- Backtest bias: 0.015784222600859966
- Under-forecast rate: 0.5063657407407407

## Planning Output

- Forecast rows generated: 3456
- Policy recommendation rows: 288
- Slotting readiness rows: 288
- High-access candidates: 235

## Runtime Publication

- Spring forecast_results publication: {'published_rows': 3456, 'model_name': 'V7_RM_PM_DIRECT'}

## Evaluation-Safe Statement

Direct RM/PM forecasting is the correct production architecture for the current data state, but the current model should remain a candidate champion until residual diagnostics, interval calibration, and per-material stability are reviewed. FG-to-RM BOM explosion remains a controlled secondary path until BOM coverage is complete and validated.


In [3]:
summary = load_json("data_lineage_summary.json")
pd.DataFrame([
    {"claim": "v7 forecasts RM/PM directly", "evidence": f"{summary.get('demand_materials')} materials, {summary.get('demand_rows')} monthly rows"},
    {"claim": "BOM explosion is not production-primary", "evidence": f"BOM product-parent coverage {summary.get('bom_product_parent_coverage_pct')}%"},
    {"claim": "Spring forecast_results is canonical", "evidence": str(summary.get("publication", "publication not requested in this run"))},
    {"claim": "LightGBM is candidate champion", "evidence": f"selected model {summary.get('selected_model')} with leaderboard in model_leaderboard.csv"},
    {"claim": "Dashboard residuals need backtest source", "evidence": "forward forecast_results rows normally have null y_true"},
])

,claim,evidence
0,v7 forecasts RM/PM directly,"288 materials, 10368 monthly rows"
1,BOM explosion is not production-primary,BOM product-parent coverage 0.0%
2,Spring forecast_results is canonical,"{'published_rows': 3456, 'model_name': 'V7_RM_..."
3,LightGBM is candidate champion,selected model lightgbm_global_rm_pm with lead...
4,Dashboard residuals need backtest source,forward forecast_results rows normally have nu...


In [4]:
load_csv("model_leaderboard.csv")

,model,rows,materials,WAPE,MAE,RMSE,Bias,under_forecast_rate
0,lightgbm_global_rm_pm,1728,288,0.250892,433.242183,2421.593928,0.015784,0.506366
1,croston_sba,1728,288,0.258377,446.166762,2401.367369,0.031150,0.434606
2,moving_avg_6,1728,288,0.282855,488.434896,2752.240619,0.082323,0.395255
3,moving_avg_3,1728,288,0.289843,500.502894,2724.766406,0.065316,0.400463
4,seasonal_naive,1728,288,0.348246,601.353009,3386.662683,0.075307,0.374421
